# Experiment 0: Image Processing Basics (Colab Version)

This notebook demonstrates basic image processing operations: image loading, RGB channel visualization, simple point operations, filtering, and gradient extraction.

The original version used a fixed Google Drive path. This cleaned version keeps the same core operations but adds a safe fallback image so the notebook can run even if the Drive image is unavailable.

## Optional: Mount Google Drive in Colab

Run the next cell only if you want to read an image from Google Drive. If the image path does not exist, the notebook automatically uses a generated demo image.

In [ ]:
# Optional for Google Colab:
# from google.colab import drive
# drive.mount('/content/drive')

## Setup: Imports, Parameters, and Helper Functions

In [ ]:

"""Experiment 0: Image Processing Basics (Colab Version).

This notebook demonstrates basic image processing operations:
1. Load and display an RGB image.
2. Display the R/G/B channels as grayscale images and 3D surfaces.
3. Apply simple point operations independently to each pixel.
4. Apply box filtering and sharpening with OpenCV filter2D.
5. Extract gradients with Sobel and Laplacian operators.

The original notebook used a fixed Google Drive image path. This version keeps the
same core demonstrations but adds a safe fallback image so that the notebook can
run even when the Drive image is not available.
"""

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d  # noqa: F401, kept for 3D plotting support

DEFAULT_IMAGE_PATH = "/content/drive/MyDrive/colab_data/zuowm.png"
TARGET_SIZE = (236, 236)  # (height, width), matching the original notebook


def make_demo_image(size=TARGET_SIZE):
    """Create a deterministic RGB demo image when no external image is found."""
    h, w = size
    yy, xx = np.mgrid[0:h, 0:w]
    x = xx.astype(np.float32) / max(w - 1, 1)
    y = yy.astype(np.float32) / max(h - 1, 1)

    red = x
    green = y
    blue = 0.5 + 0.25 * np.sin(2 * np.pi * x * 3) + 0.25 * np.cos(2 * np.pi * y * 2)
    frame = np.dstack([red, green, blue]).astype(np.float32)

    # Add several high-contrast structures so that filtering and gradients are visible.
    cv2.rectangle(frame, (25, 25), (95, 95), (1.0, 0.2, 0.2), thickness=-1)
    cv2.circle(frame, (165, 80), 35, (0.1, 0.9, 0.3), thickness=-1)
    cv2.line(frame, (30, 190), (210, 145), (0.95, 0.95, 0.05), thickness=6)
    return np.clip(frame, 0.0, 1.0).astype(np.float32)


def load_rgb_image(path=DEFAULT_IMAGE_PATH, target_size=TARGET_SIZE):
    """Load an RGB image in [0, 1]. Use a generated fallback image if needed."""
    if path and os.path.exists(path):
        img = cv2.imread(path, cv2.IMREAD_UNCHANGED)
        if img is None:
            frame = make_demo_image(target_size)
            return frame, "generated fallback image (file could not be decoded)"

        if img.ndim == 2:
            img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        elif img.shape[2] == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2RGB)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # cv2.resize expects size as (width, height).
        img = cv2.resize(img, (target_size[1], target_size[0]), interpolation=cv2.INTER_AREA)
        frame = img.astype(np.float32) / 255.0
        return np.clip(frame, 0.0, 1.0), path

    frame = make_demo_image(target_size)
    return frame, "generated fallback image (path not found)"


def show_image(frame, title="Loaded RGB Image"):
    """Display a single RGB image."""
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(1, 1, 1)
    ax.set_title(title)
    ax.imshow(np.clip(frame, 0, 1))
    ax.set_xticks([])
    ax.set_yticks([])
    return fig


def plot_rgb_channels(frame):
    """Display R, G, and B channels as grayscale images."""
    titles = ["Red Channel", "Green Channel", "Blue Channel"]
    fig = plt.figure(figsize=(12, 4))
    fig.subplots_adjust(hspace=0.4, wspace=0.2)
    for i, title in enumerate(titles):
        ax = fig.add_subplot(1, 3, i + 1)
        ax.set_title(title)
        ax.imshow(frame[:, :, i], cmap="gray", vmin=0, vmax=1)
        ax.set_xticks([])
        ax.set_yticks([])
    return fig


def plot_rgb_surfaces(frame, step=4):
    """Display downsampled R, G, and B channels as 3D intensity surfaces."""
    sampled = frame[::step, ::step, :]
    y = np.arange(sampled.shape[0])
    x = np.arange(sampled.shape[1])
    X, Y = np.meshgrid(x, y)

    print("X shape:", X.shape, "dtype:", X.dtype)
    print("Y shape:", Y.shape, "dtype:", Y.dtype)

    titles = ["Red Surface", "Green Surface", "Blue Surface"]
    fig = plt.figure(figsize=(18, 6))
    fig.subplots_adjust(hspace=0.4, wspace=0.2)
    for i, title in enumerate(titles):
        ax = fig.add_subplot(1, 3, i + 1, projection="3d")
        ax.set_title(title)
        ax.plot_surface(X, Y, sampled[:, :, i], cmap="jet", linewidth=0, antialiased=True)
    return fig


def point_operation_examples(frame):
    """Apply the same point operations as the original notebook."""
    return [
        ("Original", frame),
        ("Darken", np.clip(frame - 0.5, 0, 1)),
        ("Lower Contrast", frame / 2),
        ("Nonlinear Lower Contrast", np.power(frame, 1 / 3)),
        ("Invert", 1 - frame),
        ("Lighten", np.clip(frame + 0.5, 0, 1)),
        ("Raise Contrast", np.clip(frame * 2, 0, 1)),
        ("Nonlinear Raise Contrast", np.power(frame, 2)),
    ]


def plot_point_operations(frame):
    """Show all point-operation examples in a 2 x 4 layout."""
    examples = point_operation_examples(frame)
    fig = plt.figure(figsize=(16, 8))
    fig.subplots_adjust(hspace=0.1, wspace=0.1)
    for i, (title, image) in enumerate(examples):
        ax = fig.add_subplot(2, 4, i + 1)
        ax.set_title(title)
        ax.imshow(np.clip(image, 0, 1))
        ax.set_xticks([])
        ax.set_yticks([])
    return fig


def box_filter(frame, kernel_size=4):
    """Apply a box filter using cv2.filter2D, matching the original notebook."""
    kernel_size = int(kernel_size)
    kernel = np.ones((kernel_size, kernel_size), dtype=np.float32) / (kernel_size ** 2)
    image = cv2.filter2D(frame, -1, kernel)
    return np.clip(image, 0, 1), kernel


def sharpening_filter(frame, kernel_size=3):
    """Apply the original sharpening kernel with cv2.filter2D."""
    kernel_size = int(kernel_size)
    kernel = -np.ones((kernel_size, kernel_size), dtype=np.float32) / (kernel_size ** 2)
    kernel[(kernel_size - 1) // 2, (kernel_size - 1) // 2] += 2
    image = cv2.filter2D(frame, -1, kernel)
    return np.clip(image, 0, 1), kernel


def plot_filter_examples(frame):
    """Display the original image, box-filtered image, and sharpened image."""
    box_img, box_kernel = box_filter(frame, kernel_size=4)
    sharp_img, sharp_kernel = sharpening_filter(frame, kernel_size=3)

    fig = plt.figure(figsize=(15, 5))
    fig.subplots_adjust(wspace=0.1)
    for i, (title, image) in enumerate([
        ("Original", frame),
        ("Box Filter, N=4", box_img),
        ("Sharpening Filter, N=3", sharp_img),
    ]):
        ax = fig.add_subplot(1, 3, i + 1)
        ax.set_title(title)
        ax.imshow(np.clip(image, 0, 1))
        ax.set_xticks([])
        ax.set_yticks([])

    print("Box filter kernel:\n", box_kernel)
    print("Sharpening filter kernel:\n", sharp_kernel)
    return fig


def normalize_abs(image):
    """Normalize absolute values to [0, 1] for visualization."""
    abs_img = np.abs(image).astype(np.float32)
    max_value = float(abs_img.max())
    if max_value <= 1e-12:
        return np.zeros_like(abs_img, dtype=np.float32)
    return abs_img / max_value


def gradient_examples(frame, kernel_size=7):
    """Compute Sobel-X, Sobel-Y, and Laplacian responses."""
    kernel_size = int(kernel_size)
    laplacian = cv2.Laplacian(frame, -1, ksize=kernel_size)
    sobel_x = cv2.Sobel(frame, -1, 1, 0, ksize=kernel_size)
    sobel_y = cv2.Sobel(frame, -1, 0, 1, ksize=kernel_size)
    return sobel_x, sobel_y, laplacian


def plot_gradient_examples(frame, kernel_size=7):
    """Display gradient extraction examples."""
    sobel_x, sobel_y, laplacian = gradient_examples(frame, kernel_size=kernel_size)
    examples = [
        ("Original", frame),
        ("Sobel X", normalize_abs(sobel_x)),
        ("Sobel Y", normalize_abs(sobel_y)),
        ("Laplacian", normalize_abs(laplacian)),
    ]

    fig = plt.figure(figsize=(8, 8))
    fig.subplots_adjust(hspace=0.1, wspace=0.1)
    for i, (title, image) in enumerate(examples):
        ax = fig.add_subplot(2, 2, i + 1)
        ax.set_title(title)
        ax.imshow(np.clip(image, 0, 1), cmap="gray")
        ax.set_xticks([])
        ax.set_yticks([])
    return fig


def build_preview_figure(frame=None):
    """Build a compact preview figure for quick validation and reporting."""
    if frame is None:
        frame, _ = load_rgb_image()
    box_img, _ = box_filter(frame, kernel_size=4)
    sharp_img, _ = sharpening_filter(frame, kernel_size=3)
    sobel_x, sobel_y, laplacian = gradient_examples(frame, kernel_size=7)

    preview_items = [
        ("Original", frame),
        ("Red Channel", frame[:, :, 0]),
        ("Green Channel", frame[:, :, 1]),
        ("Blue Channel", frame[:, :, 2]),
        ("Darken", np.clip(frame - 0.5, 0, 1)),
        ("Invert", 1 - frame),
        ("Box Filter", box_img),
        ("Sharpening", sharp_img),
        ("Sobel X", normalize_abs(sobel_x)),
        ("Sobel Y", normalize_abs(sobel_y)),
        ("Laplacian", normalize_abs(laplacian)),
        ("Nonlinear Raise", np.power(frame, 2)),
    ]

    fig = plt.figure(figsize=(14, 10))
    fig.subplots_adjust(hspace=0.25, wspace=0.05)
    for i, (title, image) in enumerate(preview_items):
        ax = fig.add_subplot(3, 4, i + 1)
        ax.set_title(title)
        if image.ndim == 2:
            ax.imshow(np.clip(image, 0, 1), cmap="gray", vmin=0, vmax=1)
        else:
            ax.imshow(np.clip(image, 0, 1))
        ax.set_xticks([])
        ax.set_yticks([])
    return fig


def run_self_check():
    """Run a small validation test for the main functions."""
    frame, source = load_rgb_image(path="", target_size=TARGET_SIZE)
    assert frame.shape == (TARGET_SIZE[0], TARGET_SIZE[1], 3)
    assert frame.dtype in (np.float32, np.float64)
    assert np.isfinite(frame).all()
    assert 0.0 <= float(frame.min()) <= float(frame.max()) <= 1.0

    examples = point_operation_examples(frame)
    assert len(examples) == 8
    for _, image in examples:
        assert image.shape == frame.shape
        assert np.isfinite(image).all()

    box_img, box_kernel = box_filter(frame, kernel_size=4)
    sharp_img, sharp_kernel = sharpening_filter(frame, kernel_size=3)
    assert box_img.shape == frame.shape
    assert sharp_img.shape == frame.shape
    assert abs(float(box_kernel.sum()) - 1.0) < 1e-5
    assert sharp_kernel.shape == (3, 3)

    sobel_x, sobel_y, laplacian = gradient_examples(frame, kernel_size=7)
    assert sobel_x.shape == frame.shape
    assert sobel_y.shape == frame.shape
    assert laplacian.shape == frame.shape
    assert np.isfinite(sobel_x).all()
    assert np.isfinite(sobel_y).all()
    assert np.isfinite(laplacian).all()

    print("Self-check passed.")
    print("Image source:", source)
    print("Frame shape:", frame.shape)
    print("Frame dtype:", frame.dtype)
    print("Frame range:", float(frame.min()), "to", float(frame.max()))
    return True


## Part 1: Read and Display an Image

In [ ]:
frame, image_source = load_rgb_image(DEFAULT_IMAGE_PATH, TARGET_SIZE)
print('Image source:', image_source)
print('Frame shape:', frame.shape)
print('Frame dtype:', frame.dtype)
print('Frame range:', float(frame.min()), 'to', float(frame.max()))
show_image(frame, title='Loaded RGB Image');

### Display the R, G, and B Channels as Grayscale Images

In [ ]:
plot_rgb_channels(frame);

### Display the R, G, and B Channels as 3D Surfaces

In [ ]:
plot_rgb_surfaces(frame, step=4);

## Part 2: Point Operations

Each output pixel is computed independently from the corresponding input pixel.

In [ ]:
plot_point_operations(frame);

## Part 3: Linear Shift-Invariant Image Filtering

This part follows the original demonstration and uses `cv2.filter2D` to apply a box filter and a sharpening filter.

In [ ]:
plot_filter_examples(frame);

## Part 4: Gradient Extraction

Sobel and Laplacian filters are used to highlight image edges.

In [ ]:
plot_gradient_examples(frame, kernel_size=7);

## Quick Self-Check

In [ ]:
run_self_check()

## Compact Preview Figure

In [ ]:
preview = build_preview_figure(frame)
# preview.savefig('Experiment0_preview_validation.png', dpi=160, bbox_inches='tight')

In [ ]:
# End of Experiment 0